# Week 07 Lab — Semantic Annotation with USAS and PyMUSAS

## Learning goals
- Understand the **USAS semantic tagset** and how semantic tagging differs from POS tagging.
- Use the **online USAS tagger** to annotate texts and interpret semantic labels.
- Install and run **PyMUSAS** programmatically on a small corpus.
- Apply semantic tagging to a **non-English language** (cross-lingual).
- Extract **spatial, emotional, and temporal** entities for Spatial Humanities research.

## Part 0 — Setup

In [ ]:
!pip install -q pymusas spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
import re
from collections import Counter, defaultdict

import spacy

print("All imports OK.")

---
## Part 1 — Online USAS Tagger (Manual Exploration)

Before using PyMUSAS programmatically, explore the online tagger to build intuition for USAS labels.

**Steps:**
1. Go to the [USAS online tagger](https://ucrel-api.lancaster.ac.uk/usas/tagger.html).
2. Copy and paste the three sample texts below one by one.
3. Record the USAS tags assigned to the highlighted words in the table cells.

### Sample texts

**Text A (news):**
> The president signed a new climate agreement at the summit in Geneva, pledging to cut carbon emissions by 50% before 2035.

**Text B (travel narrative):**
> She walked slowly along the river bank, her heart heavy with grief, watching the boats drift towards the distant mountains.

**Text C (science):**
> Researchers discovered that the drug significantly reduced inflammation in patients suffering from chronic joint pain.

### Task 1A — Record tags

Fill in the table below with the USAS tags you observed.

| Text | Word/MWE | USAS Tag | Tag Description | Correct? |
|------|----------|----------|-----------------|----------|
| A    | president | | | |
| A    | climate agreement | | | |
| A    | emissions | | | |
| B    | walked | | | |
| B    | grief | | | |
| B    | drift | | | |
| C    | drug | | | |
| C    | inflammation | | | |
| C    | chronic | | | |

_Reference: [USAS Semantic Tagset PDF](https://ucrel.lancs.ac.uk/usas/USASSemanticTagset.pdf)_

### Questions
1. Give one example where the USAS tagger assigned an **incorrect or ambiguous** tag. What tag would you assign instead?
2. How does semantic tagging differ from POS tagging? What extra information does it provide?
3. Why might **multi-word expressions (MWEs)** be important for semantic tagging?

_Your answers here._

---
## Part 2 — PyMUSAS: Programmatic Semantic Tagging

PyMUSAS is a spaCy-compatible library for rule-based semantic tagging using the USAS tagset.

In [ ]:
# Download the English PyMUSAS tagger resources
!python -m spacy download en_core_web_trf -q  # or en_core_web_sm
!pip install -q pymusas_models
!python -m spacy download en_dual_none_contextual -q  # PyMUSAS English model

In [ ]:
# Coding Task A — Load PyMUSAS pipeline
# Follow: https://ucrel.github.io/pymusas/usage/how_to/tag_text

import spacy

# TODO: load the spaCy model and add the PyMUSAS tagger
# Option 1 (recommended): load 'en_dual_none_contextual' directly
# Option 2: load 'en_core_web_sm' and add pymusas rule-based tagger manually

nlp = None  # TODO: replace with the correct spaCy load call

# Test sentence
test_text = "The bank can guarantee deposits will eventually cover future tuition costs."
doc = nlp(test_text)

print(f"{'Token':<20} {'POS':<8} {'USAS Tags'}")
print("-" * 50)
for token in doc:
    # TODO: access the semantic tags — in PyMUSAS they are stored in token._.pymusas_tags
    usas_tags = []  # TODO
    print(f"{token.text:<20} {token.pos_:<8} {usas_tags}")

### Questions
1. The word **"bank"** is ambiguous (financial institution vs. river bank). What tag did PyMUSAS assign? Is it correct in this context?
2. How does PyMUSAS handle word sense disambiguation — is it rule-based or neural?

_Your answers here._

---
## Part 3 — Annotating a Small News Corpus

Apply PyMUSAS to a small corpus and analyse the distribution of semantic categories.

In [ ]:
# Small BBC-style news corpus (replace with real articles if you have them)
CORPUS = [
    """The United Nations climate summit in Dubai concluded with a landmark agreement 
    to transition away from fossil fuels. World leaders pledged billions in funding 
    to support developing nations in shifting to renewable energy sources.""",

    """Scientists have discovered a new species of deep-sea fish off the coast of 
    New Zealand. The creature, found at depths of over 3,000 metres, emits a 
    bioluminescent glow and feeds on small crustaceans.""",

    """Tech giant Apple announced record quarterly profits driven by strong iPhone 
    sales in Asia. The company also revealed plans to expand its AI research 
    division and invest heavily in chip manufacturing.""",

    """A powerful earthquake struck southern Turkey early on Tuesday, killing at 
    least 12 people and injuring hundreds. Rescue teams worked through the night 
    to pull survivors from collapsed buildings.""",

    """A new study published in The Lancet suggests that regular physical exercise 
    significantly reduces the risk of depression and anxiety. Researchers analysed 
    data from over 150,000 participants across 15 countries.""",
]


In [ ]:
# Coding Task A — Tag the entire corpus and collect USAS tags

def tag_corpus(texts: list, nlp) -> list:
    """
    Tag each text with PyMUSAS.
    Returns a list of lists: for each text, a list of (token, usas_tags) tuples.
    """
    results = []
    for text in texts:
        doc = nlp(text)
        tagged = []
        for token in doc:
            # TODO: extract token.text and token._.pymusas_tags
            tagged.append((token.text, []))  # TODO: replace [] with actual tags
        results.append(tagged)
    return results

corpus_tags = tag_corpus(CORPUS, nlp)

# Print first article's tags
print("Article 1 — first 15 tokens:")
print(f"{'Token':<20} {'Top USAS Tag'}")
print("-" * 35)
for token, tags in corpus_tags[0][:15]:
    top_tag = tags[0] if tags else "O"
    print(f"{token:<20} {top_tag}")

In [ ]:
# Coding Task B — Compute tag frequency distribution across the corpus

def top_level_category(tag: str) -> str:
    """
    Extract the top-level USAS category letter from a tag.
    e.g. 'A1.1.1' -> 'A', 'M7' -> 'M', 'Z99' -> 'Z'
    Strip leading +/- or trailing modifiers.
    """
    tag = tag.lstrip('+-')
    # TODO: return the first alphabetic character(s) before any digit
    match = re.match(r'([A-Z]+)', tag)
    return match.group(1) if match else 'UNK'

# TODO: count top-level categories across all corpus_tags
category_counts = Counter()
for article in corpus_tags:
    for token, tags in article:
        if tags:
            # TODO: add top_level_category of tags[0] to category_counts
            pass

print("Top 15 USAS semantic categories in corpus:")
print(f"{'Category':<12} {'Count':>6}")
print("-" * 20)
for cat, count in category_counts.most_common(15):
    print(f"{cat:<12} {count:>6}")

### Questions
1. Which semantic categories dominate your corpus? Does this match your expectation for news text?
2. How would the tag distribution differ if you used a corpus of poetry or medical records?
3. What is a **hapax legomenon** and how might rare tags affect analysis?

_Your answers here._

---
## Part 4 — Cross-Lingual Semantic Tagging

PyMUSAS supports multiple languages. Tag a short Spanish text to explore the semantic categories without translating.

In [ ]:
# Download Spanish PyMUSAS resources
!python -m spacy download es_core_news_sm -q
# Spanish PyMUSAS single tagger
!python -m spacy download es_single_none_contextual -q

In [ ]:
# Coding Task A — Load Spanish PyMUSAS pipeline

# TODO: load 'es_single_none_contextual' (or es_core_news_sm + add pymusas tagger)
nlp_es = None  # TODO

spanish_text = """
El presidente del gobierno anunció nuevas medidas para combatir el cambio climático.
Los científicos advierten que el aumento de las temperaturas provoca inundaciones 
y sequías en todo el mundo. Las ciudades costeras están en grave peligro.
"""

doc_es = nlp_es(spanish_text)

print(f"{'Token':<25} {'POS':<8} {'USAS Tags'}")
print("-" * 55)
for token in doc_es:
    if not token.is_space:
        usas_tags = []  # TODO: token._.pymusas_tags
        print(f"{token.text:<25} {token.pos_:<8} {usas_tags}")

In [ ]:
# Coding Task B — Guess the main topics from USAS tags WITHOUT translating

# USAS Category reference (partial):
USAS_CATEGORIES = {
    'A': 'General and abstract terms',
    'B': 'The body and the individual',
    'C': 'Arts and crafts',
    'E': 'Emotion',
    'F': 'Food and farming',
    'G': 'Government and the public domain',
    'H': 'Architecture, housing and the home',
    'I': 'Money and commerce',
    'K': 'Entertainment and sport',
    'L': 'Life and living things',
    'M': 'Movement, location, travel and transport',
    'N': 'Numbers and measurement',
    'O': 'Substances, materials, objects and equipment',
    'P': 'Education',
    'Q': 'Language and communication',
    'S': 'Social actions, states and processes',
    'T': 'Time',
    'W': 'The world and our environment',
    'X': 'Psychological actions, states and processes',
    'Y': 'Science and technology',
    'Z': 'Names and grammar',
}

# TODO: count top-level categories in doc_es and map to descriptions
es_categories = Counter()
for token in doc_es:
    if not token.is_space:
        tags = []  # TODO: token._.pymusas_tags
        if tags:
            cat = top_level_category(tags[0])
            es_categories[cat] += 1

print("Top USAS categories in Spanish text:")
for cat, count in es_categories.most_common(10):
    desc = USAS_CATEGORIES.get(cat, 'Unknown')
    print(f"  {cat:<5} {count:>3}  —  {desc}")

print("\nBased on these categories alone, what do you think the text is about?")
print("→ YOUR GUESS: ___")

### Questions
1. Were you able to identify the main topics of the Spanish text from USAS tags alone? What helped most?
2. What are the limitations of cross-lingual semantic tagging with rule-based systems?
3. How might a neural multilingual model (e.g., mBERT) handle semantic tagging differently?

_Your answers here._

---
## Part 5 — Spatial Humanities: Extracting E, M, T Entities

In the [Spatial Humanities project](https://spacetimenarratives.github.io/), semantic tagging is used to extract:
- **E** — Emotion tags (e.g., `E4.1`, `E4.2`)
- **M** — Movement, location, travel and transport (e.g., `M1`, `M6`, `M7`, `M8`)
- **T** — Time (e.g., `T1.1`, `T1.2`, `T2`, `T3`)

These are used to answer questions about spatial entities and narratives in historical texts.

In [ ]:
# Travel narrative corpus
NARRATIVES = [
    """She departed at dawn, moving swiftly northward along the winding river path. 
    The journey filled her with dread and longing as she left her childhood home behind.
    By midday she had reached the ancient bridge, exhausted but hopeful.""",

    """The soldiers marched through the valley for three days, arriving at the fortress 
    walls at sunset. Their hearts were heavy with fear, yet they pressed forward with 
    determination. The commander ordered a halt near the eastern gate.""",

    """In the summer of 1842, the expedition sailed south along the coast, passing 
    small fishing villages and rocky headlands. The crew grew restless after weeks 
    at sea, desperate to set foot on land once more.""",
]


In [ ]:
# Coding Task A — Extract EMT tokens

# USAS category prefixes for each type
EMOTION_CATS    = {'E'}           # Emotion
MOVEMENT_CATS   = {'M'}           # Movement, location, travel
TIME_CATS       = {'T'}           # Time

def extract_emt(text: str, nlp) -> dict:
    """
    Tag text with PyMUSAS and extract tokens belonging to E, M, or T categories.
    Returns dict:
      {
        'emotion'  : list of (token_text, usas_tag),
        'movement' : list of (token_text, usas_tag),
        'time'     : list of (token_text, usas_tag),
      }
    """
    doc = nlp(text)
    result = {'emotion': [], 'movement': [], 'time': []}

    for token in doc:
        tags = []  # TODO: token._.pymusas_tags
        if not tags:
            continue
        top = tags[0]
        cat = top_level_category(top)

        # TODO: append (token.text, top) to the correct result list based on cat
        pass

    return result


for i, narrative in enumerate(NARRATIVES, 1):
    emt = extract_emt(narrative, nlp)
    print(f"\n=== Narrative {i} ===")
    print(f"Emotion  ({len(emt['emotion'])}): {[t for t, _ in emt['emotion']]}")
    print(f"Movement ({len(emt['movement'])}): {[t for t, _ in emt['movement']]}")
    print(f"Time     ({len(emt['time'])}): {[t for t, _ in emt['time']]}")

In [ ]:
# Coding Task B — Compute EMT density per narrative

def emt_density(text: str, nlp) -> dict:
    """
    Compute the proportion of tokens in each EMT category.
    Returns dict: {'emotion': float, 'movement': float, 'time': float}
    """
    doc = nlp(text)
    total_tokens = sum(1 for t in doc if not t.is_space and not t.is_punct)
    emt = extract_emt(text, nlp)

    # TODO: compute density = len(emt[category]) / total_tokens
    return {
        'emotion':  0.0,  # TODO
        'movement': 0.0,  # TODO
        'time':     0.0,  # TODO
    }

print(f"{'Narrative':<12} {'Emotion':>10} {'Movement':>10} {'Time':>10}")
print("-" * 45)
for i, narrative in enumerate(NARRATIVES, 1):
    density = emt_density(narrative, nlp)
    print(f"Narrative {i}  {density['emotion']:>10.3f} {density['movement']:>10.3f} {density['time']:>10.3f}")

In [ ]:
# Coding Task C — Direction words in Movement category

DIRECTION_WORDS = {
    'north', 'south', 'east', 'west', 'northward', 'southward',
    'eastward', 'westward', 'upstream', 'downstream', 'inland', 'coastal',
    'forward', 'backward', 'upward', 'downward', 'along', 'across',
    'toward', 'towards', 'away', 'beyond', 'through', 'past',
}

def extract_directions(text: str, nlp) -> list:
    """
    Extract direction/movement words from text.
    Returns list of (token_text, usas_tag, sentence_context).
    Include both DIRECTION_WORDS matches and Movement-tagged tokens.
    """
    doc = nlp(text)
    directions = []

    for sent in doc.sents:
        for token in sent:
            tags = []  # TODO: token._.pymusas_tags
            top_cat = top_level_category(tags[0]) if tags else ''
            is_direction = (
                token.text.lower() in DIRECTION_WORDS
                or top_cat == 'M'
            )
            if is_direction:
                usas = tags[0] if tags else 'O'
                directions.append((token.text, usas, sent.text.strip()))

    return directions

for i, narrative in enumerate(NARRATIVES, 1):
    dirs = extract_directions(narrative, nlp)
    print(f"\n=== Narrative {i} — Direction/Movement Tokens ===")
    for tok, tag, ctx in dirs:
        print(f"  '{tok}' [{tag}]")

### Questions
1. How could EMT density scores be used to compare different types of narrative (e.g., adventure vs. romance novels)?
2. What information is lost when we only keep the top-level USAS category (e.g., `M`) rather than the full tag (e.g., `M8`)?
3. Describe one research question in the Spatial Humanities that could be answered using the EMT extraction pipeline you built.

_Your answers here._

---
## Part 6 — Keyword Analysis with USAS

Compare semantic tag distributions across two corpora to identify semantically distinctive categories (keyness).

In [ ]:
# Two mini-corpora for comparison
CORPUS_A = NARRATIVES  # travel/emotion narratives
CORPUS_B = CORPUS      # news articles (from Part 3)

In [ ]:
# Coding Task A — Build tag frequency profiles for each corpus

def build_tag_profile(texts: list, nlp) -> Counter:
    """
    Return a Counter of top-level USAS categories across all texts.
    Ignore punctuation and spaces.
    """
    counts = Counter()
    for text in texts:
        doc = nlp(text)
        for token in doc:
            if token.is_space or token.is_punct:
                continue
            tags = []  # TODO: token._.pymusas_tags
            if tags:
                # TODO: increment counts[top_level_category(tags[0])]
                pass
    return counts

profile_a = build_tag_profile(CORPUS_A, nlp)
profile_b = build_tag_profile(CORPUS_B, nlp)

# Normalise to relative frequencies
total_a = sum(profile_a.values()) or 1
total_b = sum(profile_b.values()) or 1

all_cats = set(profile_a) | set(profile_b)

print(f"{'Category':<8} {'Desc':<35} {'Corpus A %':>10} {'Corpus B %':>10} {'Ratio A/B':>10}")
print("-" * 80)
rows = []
for cat in all_cats:
    freq_a = profile_a.get(cat, 0) / total_a
    freq_b = profile_b.get(cat, 0) / total_b
    ratio = freq_a / freq_b if freq_b > 0 else float('inf')
    desc = USAS_CATEGORIES.get(cat, 'Other')
    rows.append((cat, desc, freq_a, freq_b, ratio))

# Sort by ratio descending (most distinctive to Corpus A first)
for cat, desc, fa, fb, ratio in sorted(rows, key=lambda r: -r[4])[:15]:
    print(f"{cat:<8} {desc:<35} {fa*100:>10.2f} {fb*100:>10.2f} {ratio:>10.2f}")

### Questions
1. Which USAS categories most strongly distinguish the narrative corpus from the news corpus? Does this make linguistic sense?
2. What is **keyness** in corpus linguistics, and how does the ratio you computed relate to it?
3. How would you make this comparison more statistically rigorous (name one statistical test)?

_Your answers here._

---
## Submission
- Complete all `TODO` sections.
- Answer all written questions in the markdown cells above.
- Include the tag frequency table from Part 3B and the EMT density table from Part 5B.
- Submit your completed `.ipynb` notebook as instructed.